In [ ]:
import os
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import torch

!pip install seqeval
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=0ac8cba824c143adcdfca6618c3e6930c4dae7c4bc77d5ce0e197fd28a146f1c
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
MODEL_NAME = "bert-base-cased"
DATASET_NAME = "mnaguib/WikiNER"
DATASET_CONFIG = "en" #tambahan biar ga error
OUTPUT_DIR = "./bert_ner_output"
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
SEED = 42

In [ ]:
LABEL_LIST = ["O", "LOC", "PER", "MISC", "ORG"]
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}
NUM_LABELS = len(LABEL_LIST)

In [ ]:
def load_wikiner():
    """
    Load WikiNER from Hugging Face.
    Expected columns: tokens (list of str), ner_tags (list of int or str).
    """
    print("Loading WikiNER dataset....")
    dataset = load_dataset(DATASET_NAME, DATASET_CONFIG) #tambahan
    print(dataset)
    return dataset

dataset = load_wikiner()

Loading WikiNER dataset....


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/1.95k [00:00<?, ?B/s]

data/en/train.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

data/en/test.parquet:   0%|          | 0.00/1.57M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/129376 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/14398 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'words', 'ner_tags'],
        num_rows: 129376
    })
    test: Dataset({
        features: ['id', 'words', 'ner_tags'],
        num_rows: 14398
    })
})


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_and_align_labels(examples):
    """
    Tokenize the input tokens and align NER labels to the new subword tokens.

    Key challenge: BERT uses WordPiece, so one word can become multiple subword
    tokens. We assign:
      - the correct label to the FIRST subword of each word
      - -100 to subsequent subwords (ignored by the loss function)
    """
    tokenized_inputs = tokenizer(
        examples["words"],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True,   # input is already tokenized at word level
        padding=False,              # padding handled by DataCollator
    )

    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)
            previous_word_id = word_id

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,  # drop original columns
)
print("Tokenization complete.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Tokenizing dataset...


Map:   0%|          | 0/129376 [00:00<?, ? examples/s]

Map:   0%|          | 0/14398 [00:00<?, ? examples/s]

Tokenization complete.


In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

print(f"\nModel loaded: {MODEL_NAME}")
print(f"Total parameters: {model.num_parameters():,}")
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}\n")

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i


Model loaded: bert-base-cased
Total parameters: 107,723,525
Trainable parameters: 107,723,525



In [ ]:
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LEARNING_RATE,
    weight_decay                = WEIGHT_DECAY,
    warmup_ratio                = 0.1,
    eval_strategy         = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    logging_dir                 = f"{OUTPUT_DIR}/logs",
    logging_steps               = 100,
    seed                        = SEED,
    report_to                   = "none",
    fp16                        = torch.cuda.is_available(),
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
#tambahan biar ga error
from datasets import DatasetDict

print("Memecah dataset menjadi Train, Validation, dan Test...")

#80% buat Train, 20% buat cadangan
train_test_split = tokenized_dataset["train"].train_test_split(test_size=0.2, seed=SEED)

# cadangan 20% tadi jadi: 10% Validation, 10% Test
val_test_split = train_test_split["test"].train_test_split(test_size=0.5, seed=SEED)

#gabungin ke satu wadah anamanya tokenized_dataset
tokenized_dataset = DatasetDict({
    "train": train_test_split["train"],
    "validation": val_test_split["train"],
    "test": val_test_split["test"]
})

print("Berhasil! Ukuran dataset sekarang:")
print(tokenized_dataset)

Memecah dataset menjadi Train, Validation, dan Test...
Berhasil! Ukuran dataset sekarang:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 103500
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 12938
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 12938
    })
})


In [ ]:
# ── 9. Trainer ────────────────────────────────────────────────────────────────

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (where label == -100)
    true_labels = [[ID2LABEL[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [ID2LABEL[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Calculate metrics
    precision = precision_score(true_labels, true_predictions)
    recall = recall_score(true_labels, true_predictions)
    f1 = f1_score(true_labels, true_predictions)

    return {"precision": precision, "recall": recall, "f1": f1}

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized_dataset["train"],
    eval_dataset    = tokenized_dataset["validation"],
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
)

In [ ]:
print("=" * 60)
print("Starting BERT fine-tuning for NER...")
print("=" * 60)

trainer.train()

Starting BERT fine-tuning for NER...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.033819,0.033075,0.946970,0.942249,0.944604
2,0.022829,0.029696,0.943441,0.956231,0.949793
3,0.014413,0.034182,0.945205,0.957386,0.951256


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LOC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ORG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PER seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LOC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ORG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PER seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LOC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ORG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PER seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=19407, training_loss=0.036243206275795074, metrics={'train_runtime': 1511.719, 'train_samples_per_second': 205.395, 'train_steps_per_second': 12.838, 'total_flos': 9547693368309720.0, 'train_loss': 0.036243206275795074, 'epoch': 3.0})

In [ ]:
print("\n" + "=" * 60)
print("Evaluating on TEST set...")
print("=" * 60)

test_results = trainer.evaluate(eval_dataset=tokenized_dataset["test"])
print(f"\nTest Results:")
print(f"  Precision : {test_results['eval_precision']:.4f}")
print(f"  Recall    : {test_results['eval_recall']:.4f}")
print(f"  F1-score  : {test_results['eval_f1']:.4f}")


Evaluating on TEST set...


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LOC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PER seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ORG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Training Loss,Validation Loss,Epoch,Precision,Recall,F1
0.014413,0.032869,3,0.945941,0.960069,0.952953



Test Results:
  Precision : 0.9459
  Recall    : 0.9601
  F1-score  : 0.9530


In [ ]:
def detailed_report(trainer, dataset_split):
    """Print per-class Precision / Recall / F1 for PER, LOC, ORG."""
    predictions_output = trainer.predict(dataset_split)
    logits = predictions_output.predictions
    label_ids = predictions_output.label_ids
    preds = np.argmax(logits, axis=-1)

    true_labels, true_preds = [], []
    for pred_seq, label_seq in zip(preds, label_ids):
        tl, tp = [], []
        for p, l in zip(pred_seq, label_seq):
            if l == -100:
                continue
            tl.append(ID2LABEL[l])
            tp.append(ID2LABEL[p])
        true_labels.append(tl)
        true_preds.append(tp)

    print("\nDetailed Per-Class Report:")
    print(classification_report(true_labels, true_preds))

detailed_report(trainer, tokenized_dataset["test"])


Detailed Per-Class Report:
              precision    recall  f1-score   support

          ER       0.96      0.97      0.97      8735
          OC       0.92      0.94      0.93      7468

   micro avg       0.95      0.96      0.95     16203
   macro avg       0.94      0.96      0.95     16203
weighted avg       0.95      0.96      0.95     16203



In [ ]:
print(f"\nSaving model to {OUTPUT_DIR}/final_model ...")
trainer.save_model(f"{OUTPUT_DIR}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_model")
print("Model saved.")


Saving model to ./bert_ner_output/final_model ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved.


In [ ]:
def predict_ner(text: str, model_path: str = f"{OUTPUT_DIR}/final_model"):
    """
    Run NER inference on a raw text string.

    Example:
        predict_ner("King Henry VIII ruled England from 1509 to 1547.")
    """
    from transformers import pipeline

    ner_pipeline = pipeline(
        "ner",
        model=model_path,
        tokenizer=model_path,
        aggregation_strategy="simple",   # merge subword tokens into full entities
    )
    results = ner_pipeline(text)

    print(f"\nInput : {text}")
    print("Entities found:")
    for ent in results:
        print(f"  [{ent['entity_group']:5s}]  '{ent['word']}'  "
              f"(score: {ent['score']:.3f})")
    return results

In [ ]:
# 1. Buka jalur ke Google Drive (kalau belum di-mount)
from google.colab import drive
drive.mount('/content/drive')

# 2. Lempar SATU FOLDER UTUH model BERT langsung ke Google Drive
!cp -r ./bert_ner_output/final_model /content/drive/MyDrive/BERT_NER_Final2

ValueError: mount failed

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

# ==========================================
# 1. GENERATE CONFUSION MATRIX
# ==========================================
print("Membuat Confusion Matrix...")
# Ambil hasil prediksi dari test set
predictions_output = trainer.predict(tokenized_dataset["test"])
preds = np.argmax(predictions_output.predictions, axis=-1)
label_ids = predictions_output.label_ids

flat_true, flat_preds = [], []
for pred_seq, label_seq in zip(preds, label_ids):
    for p, l in zip(pred_seq, label_seq):
        if l != -100: # Abaikan token -100
            flat_true.append(ID2LABEL[l])
            flat_preds.append(ID2LABEL[p])

# Plotting
cm = confusion_matrix(flat_true, flat_preds, labels=LABEL_LIST)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_LIST, yticklabels=LABEL_LIST)

# ---> PERHATIAN: GANTI JUDUL DI BAWAH INI SESUAI NAMA COLAB-NYA <---
plt.title('Confusion Matrix - BERT (Test Set)', fontsize=14)
# Kalau di Colab XLM-RoBERTa, ganti jadi: 'Confusion Matrix - XLM-RoBERTa (Test Set)'

plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.show()

# ==========================================
# 2. PERBANDINGAN TRAINING VS TESTING LOSS
# ==========================================
print("\nMengekstrak metrik Training vs Testing...")
log_history = trainer.state.log_history

train_loss = [log['loss'] for log in log_history if 'loss' in log]
eval_loss = [log['eval_loss'] for log in log_history if 'eval_loss' in log]
epochs = range(1, len(eval_loss) + 1)

# Plot Kurva Loss
plt.figure(figsize=(8, 5))

# Sinkronisasi panjang data log (karena train loss di-log per step, bukan per epoch)
if len(train_loss) > len(eval_loss):
    step_size = len(train_loss) // len(eval_loss)
    train_loss = [train_loss[i * step_size] for i in range(len(eval_loss))]

plt.plot(epochs, train_loss, 'b-', label='Training Loss', marker='o')
plt.plot(epochs, eval_loss, 'r-', label='Validation/Testing Loss', marker='s')

# ---> PERHATIAN: GANTI JUDUL DI BAWAH INI JUGA <---
plt.title('Training vs Evaluation Loss - BERT', fontsize=14)
# Kalau di Colab XLM-RoBERTa, ganti jadi: 'Training vs Evaluation Loss - XLM-RoBERTa'

plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True)
plt.show()